# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [28]:
ПРІЗВИЩЕ_ІМЯ = 'Олійник Божена'      
ГРУПА        = 'ШІ-33'
НОМЕР        = 7       

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [29]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 7   ·   Олійник Божена   ·   ШІ-33

Ваші вісім стовпців:
   1. ПІП
   2. помер/жив
   3. ступінь КТ
   4. Дата розтину
   5. КТ
   6. Т вище 37,5
   7. Сечов1
   8. БРпр1

Ваша пара для аналізу зв’язку: Стать та помер/жив

Ваше додаткове питання:
   Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.


In [30]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,помер/жив,ступінь КТ,Дата розтину,КТ,"Т вище 37,5",Сечов1,БРпр1
0,Федорюк Петро Юрійович,1,сердній,2022-03-11,20-30%,3,5.1,3.4
1,Загул Григорій Сергійович,2,середній,NaT,30-35%,3,6.6,2.8
2,Бондара Лариса Павлівна,2,критичний,NaT,70-80%,3,7.0,2.8
3,Ватаманюка Уляна Дмитрівна,2,критичний,NaT,0.8,6,9.5,3.4
4,Бабійа Лариса Олегівна,2,NaN,NaT,NaN,5 до гос,5.2,3.4
5,Ковалика ганна Андріївна,1,середній,2021-12-26,30-40%,NaN,14.0,3.4
6,Височан Віктор Володимирович,2,важкий,NaT,60-65%,NaN,5.7,5.1
7,Дзюбаа Ірина Іванівна,2,легкий,NaT,до 17%,NaN,9.1,5.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [31]:
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    object        
 1   помер/жив     121 non-null    int64         
 2   ступінь КТ    42 non-null     object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   КТ            44 non-null     object        
 5   Т вище 37,5   78 non-null     object        
 6   Сечов1        117 non-null    float64       
 7   БРпр1         109 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 7.7+ KB


In [32]:
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5
    непорожніх 78 з 121 | унікальних 35 | типи: str×48, int×30
    приклади: ['3', '6', "'5 до гос'", "'3 дні в стац'"]

--- Сечов1
    непорожніх 117 з 121 

**Що видно з огляду.** pandas показує текстовий тип (object/str) у чотирьох стовпцях, але причини різні:
ПІП — справді текст, ступінь КТ — вибір 5 враіантів з друкарськими помилками (12 написань
замість 5 варіантів), КТ — суміш часток (0.3), діапазонів (20-30%'), межі (до 17%) і порожнього
рядка, Т вище 37,5 — суміш чисел і фраз на кшталт 3 дні в стац. 
Тому тип даних шкали не визначає.

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [33]:
ПАСПОРТ = {
    'ПІП': ('ідентифікатор',
            'ПІП — позначка особи, а не величина: 121 унікальне (всі унікальні) '),
    'помер/жив': ('номінальна',
            'Коди 1 і 2 — позначення двох протилежних станів (1 — помер, 2 — жив); порядку і відстані між ними нема, можна лише перевірити на рівність.'),
    'ступінь КТ': ('порядкова',
            'Рівні легкий < середній < важкий < вкрай важкий < критичний мають природний порядок, але відстані між сусідніми ступенями невідомі й нерівні.'),
    'Дата розтину': ('інтервальна',
            'Календарна дата: різниці між датами у днях мають сенс, але відношення сенс не мають, оскільки нема нуля'),
    'КТ': ('відношень',
            'Частка уражених легень: є нуль (без уражень), 40 % — удвічі більше за 20 %, записи змішані (частки, діапазони, текст).'),
    'Т вище 37,5': ('відношень',
            'Кількість днів із температурою понад 37,5: є нуль (жодного дня), відношення мають сенс (6 днів — удвічі більше за 3), частина записів — текст.'),
    'Сечов1': ('відношень',
            'Концентрація сечовини в крові (ммоль/л) першого вимірювання: є нуль (відсутність), відношення мають сенс (60 — у вісім разів більше за 7,5).'),
    'БРпр1': ('відношень',
            'Прямий білірубін (мкмоль/л): є нуль, відношення мають сенс, значення переважно кратні 1,7 (3,4; 5,1; 6,8) — це крок округлення лабораторії, а не шкала.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"ПІП — позначка особи, а не величина: 121 унікальне (всі ..."
1,помер/жив,номінальна,Коди 1 і 2 — позначення двох протилежних станів (1 — пом...
2,ступінь КТ,порядкова,Рівні легкий < середній < важкий < вкрай важкий < критич...
3,Дата розтину,інтервальна,"Календарна дата: різниці між датами у днях мають сенс, а..."
4,КТ,відношень,"Частка уражених легень: є нуль (без уражень), 40 % — удв..."
5,"Т вище 37,5",відношень,"Кількість днів із температурою понад 37,5: є нуль (жодно..."
6,Сечов1,відношень,Концентрація сечовини в крові (ммоль/л) першого вимірюва...
7,БРпр1,відношень,"Прямий білірубін (мкмоль/л): є нуль, відношення мають се..."


In [34]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


### Приведення до коректних типів (додатковий крок)

Шкала — це зміст величини, а в файлі вона записана «як вийшло». Щоб рахувати допустимі
статистики, спершу прибираємо **кодування**, не змінюючи змісту. Оригінал `df` не чіпаємо,
працюємо з копією `дані`.

* `помер/жив`: 1 → «помер», 2 → «жив». Кодування виведено з даних: усі 61 рядки з кодом 1 мають
  дату розтину, усі 60 з кодом 2 — не мають (перевіряється `assert`-ом).
* `ступінь КТ`: 12 написань → 5 впорядкованих рівнів.
* `Стать` (потрібна для пари): `ж`/`жін` → `ж`, `ч`/`чол` → `ч`.
* `КТ`: у частку 0–1. Діапазон замінюємо серединою, «до 17%» — самим 0,17; це наближення, його
  частка фіксується в стовпці `КТ спосіб`.
* `Т вище 37,5`: з тексту беремо числа й підсумовуємо (тривалість у днях). Де саме були ці дні
  (вдома чи в стаціонарі), для 30 «голих» чисел невідомо — цю деталь губимо.

In [35]:
import re

дані = df.copy()

assert df.loc[df['помер/жив'] == 1, 'Дата розтину'].notna().all()
assert df.loc[df['помер/жив'] == 2, 'Дата розтину'].isna().all()
дані['помер/жив'] = df['помер/жив'].map({1: 'помер', 2: 'жив'})

ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
ВИПРАВ = {'легка': 'легкий', 'сердній': 'середній',
          'серед ступ': 'середній', 'серед ступеня': 'середній'}
з = df['ступінь КТ'].str.strip().replace(ВИПРАВ)
assert set(з.dropna()) <= set(ПОРЯДОК_КТ), set(з.dropna()) - set(ПОРЯДОК_КТ)
дані['ступінь КТ'] = pd.Categorical(з, categories=ПОРЯДОК_КТ, ordered=True)

дані['Стать'] = df['Стать'].replace({'жін': 'ж', 'чол': 'ч'})
assert set(дані['Стать'].dropna()) == {'ж', 'ч'}

def розібрати_КТ(v):
    if isinstance(v, (int, float, np.integer, np.floating)):
        return (np.nan, 'немає') if pd.isna(v) else (float(v), 'точно')
    t = re.sub(r'[\s%]', '', str(v))
    if t == '':
        return np.nan, 'немає'
    if t.startswith('до'):
        return float(t[2:].replace(',', '.')) / 100, 'верхня межа'
    a, b = t.replace(',', '.').split('-')
    return (float(a) + float(b)) / 200, 'діапазон'


кт = df['КТ'].map(розібрати_КТ)
дані['КТ частка'] = кт.map(lambda x: x[0])
дані['КТ спосіб'] = кт.map(lambda x: x[1])


def розібрати_дні(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    числа = re.findall(r'\d+', str(v))
    return float(sum(map(int, числа))) if числа else np.nan


дані['Т дні'] = df['Т вище 37,5'].map(розібрати_дні)


ЧИСТИЙ = {'КТ': 'КТ частка', 'Т вище 37,5': 'Т дні'}

N_СТУПІНЬ_СИРИХ = df['ступінь КТ'].nunique()
N_СТАТЬ_СИРИХ = df['Стать'].nunique()
N_КТ_НЕПОР = df['КТ'].notna().sum()
N_КТ_ТЕКСТ = int(df['КТ'].map(lambda v: isinstance(v, str)).sum())
N_Т_НЕПОР = df['Т вище 37,5'].notna().sum()
N_Т_ТЕКСТ = int(df['Т вище 37,5'].map(lambda v: isinstance(v, str)).sum())
N_СТУПІНЬ_ПРОПУСКІВ = int(df['ступінь КТ'].isna().sum())

print('ступінь КТ :', N_СТУПІНЬ_СИРИХ, 'написань →', дані['ступінь КТ'].nunique(), 'рівнів')
print(дані['ступінь КТ'].value_counts(sort=False).to_string(), '\n')
print('Стать      :', N_СТАТЬ_СИРИХ, 'написання →', дані['Стать'].nunique(), 'значення')
print('КТ         : текстових записів', N_КТ_ТЕКСТ, 'з', N_КТ_НЕПОР, '| способи запису:',
      дані['КТ спосіб'].value_counts().to_dict())
print('Т вище 37,5: текстових записів', N_Т_ТЕКСТ, 'з', N_Т_НЕПОР)

ступінь КТ : 12 написань → 5 рівнів
ступінь КТ
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3 

Стать      : 4 написання → 2 значення
КТ         : текстових записів 17 з 44 | способи запису: {'немає': 78, 'точно': 27, 'діапазон': 15, 'верхня межа': 1}
Т вище 37,5: текстових записів 48 з 78


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [36]:
def допустима_статистика(стовпець, шкала):
    з = дані[ЧИСТИЙ.get(стовпець, стовпець)].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    ЗВ = {'номінальна': 1, 'порядкова': 2, 'інтервальна': 3, 'відношень': 4, 'абсолютна': 4}
    рівень = ЗВ[ш]
    частоти = з.value_counts()
    р['мода'] = з.mode().tolist()[:3]
    р['частоти'] = (частоти[частоти > 0].to_dict() if з.nunique() <= 12
                    else 'забагато різних значень; найчастіші: ' + str(частоти.head(3).to_dict()))

    if рівень >= 2:
        впорядк = з.sort_values().reset_index(drop=True)
        k = lambda q: впорядк.iloc[int(round(q * (len(впорядк) - 1)))]
        р['медіана'] = k(0.5)
        р['квантилі 25%/75%'] = (k(0.25), k(0.75))

    if рівень >= 3:
        if pd.api.types.is_datetime64_any_dtype(з):
            р['середнє'] = з.mean().date()
            р['ст. відхилення, днів'] = round(з.std().total_seconds() / 86400, 1)
        else:
            р['середнє'] = round(з.mean(), 3)
            р['ст. відхилення'] = round(з.std(), 3)

    if рівень >= 4:
        р['мін / макс'] = (з.min(), з.max())
        р['коефіцієнт варіації'] = round(з.std() / з.mean(), 3)
    return р


for c in МОЇ:
    print(f'{c}  ({ПАСПОРТ[c][0]})')
    for k, v in допустима_статистика(c, ПАСПОРТ[c][0]).items():
        print(f'    {k:<22} {v}')
    print()

ПІП  (ідентифікатор)
    непорожніх             121
    унікальних             121

помер/жив  (номінальна)
    непорожніх             121
    унікальних             2
    мода                   ['помер']
    частоти                {'помер': 61, 'жив': 60}

ступінь КТ  (порядкова)
    непорожніх             42
    унікальних             5
    мода                   ['середній']
    частоти                {'середній': 18, 'важкий': 10, 'легкий': 7, 'вкрай важкий': 4, 'критичний': 3}
    медіана                середній
    квантилі 25%/75%       ('середній', 'важкий')

Дата розтину  (інтервальна)
    непорожніх             61
    унікальних             50
    мода                   [Timestamp('2022-06-07 00:00:00')]
    частоти                забагато різних значень; найчастіші: {Timestamp('2022-06-07 00:00:00'): 3, Timestamp('2022-04-27 00:00:00'): 2, Timestamp('2022-04-07 00:00:00'): 2}
    медіана                2022-04-23 00:00:00
    квантилі 25%/75%       (Timestamp('2022-03-29 00:

**Коментар до етапу 2.**

* ПІП — лише кількість непорожніх і унікальних: 121 з 121, тож «мода» не існує (кожне значення зустрічається один раз) — це та сама «частота», з якої нічого не прочитати.
* помер/жив — лише мода й частоти: 61 помер, 60 жив. Середнього тут немає.
* ступінь КТ — мода й частоти, медіана та квантилі. Середнього й стандартного відхилення немає: відстань від легкого до середньонго не дорівнює відстані від важкого до критичного.
* Дата розтину — середня дата й стандартне відхилення в днях допустимі, а коефіцієнт варіації — ні: нуль календаря умовний. Заповнена лише для померлих.
* КТ, Т вище 37,5, Сечов1, БРпр1 — усе, включно з коефіцієнтом варіації, але для КТ це наближення (частину значень ми замінили серединою діапазону), а для Сечов1 середнє (8,9) сильно вище медіани (7,2) через кілька великих значень (від 25 до 68).

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [37]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,1.0,1.0,1.0,2.0,2.0,0.502062
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Т вище 37,5",78.0,35.0,3.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов1,117.0,NaN,NaN,NaN,8.93188,3.1,5.5,7.2,9.5,68.0,8.174368
БРпр1,109.0,NaN,NaN,NaN,4.801468,2.4,3.4,5.1,5.3,10.2,1.575191


In [38]:
середнє_кодів = df['помер/жив'].mean()

коди_КТ = дані['ступінь КТ'].cat.codes.where(lambda s: s >= 0) + 1
середній_ранг = коди_КТ.mean()

d = df['Дата розтину'].dropna()
серійні_excel = (d - pd.Timestamp('1899-12-30')).dt.days
серійні_unix = (d - pd.Timestamp('1970-01-01')).dt.days
cv_excel = серійні_excel.std() / серійні_excel.mean()
cv_unix = серійні_unix.std() / серійні_unix.mean()

т_числа = pd.to_numeric(df['Т вище 37,5'], errors='coerce')
середнє_т = т_числа.mean()
втрачено_т = int(df['Т вище 37,5'].notna().sum() - т_числа.notna().sum())

опис_піп = df['ПІП'].describe()

БЕЗГЛУЗДІ = [
    ('помер/жив', 'середнє', round(середнє_кодів, 3),
     'Коди 1 і 2 — мітки станів, а не кількості. Результат 1,496 лише повторює частку '
     'кодів «2» і не означає ні «пів-живий», ні «середній стан»; змінивши коди на 0/1 чи 5/9, '
     'отримаємо зовсім інше середнє для тих самих людей.'),
    ('ступінь КТ (коди 1–5)', 'середнє', round(середній_ранг, 2),
     'Ступінь — порядкова шкала: відстань від легкого до середньонго не дорівнює відстані від '
     'важкого до критичного , тож додавати й ділити номери рівнів не можна; «2,9 ступеня» не існує. '
     'Коректно — медіана як рівень.'),
    ('Дата розтину', 'коефіцієнт варіації (σ/μ)', round(cv_excel, 5),
     f'Дата — інтервальна шкала з умовним нулем. Якщо відрахувати дні від нуля Excel, CV = '
     f'{cv_excel:.5f}, а від 1970 року — {cv_unix:.5f}: те саме, а число різне, бо '
     f'змінили лише точку відліку. Відношення й CV мають сенс лише для природного нуля.'),
    ('Т вище 37,5', 'середнє після to_numeric(errors="coerce")', round(середнє_т, 2),
     f'Із {int(df["Т вище 37,5"].notna().sum())} непорожніх записів {втрачено_т} були текстом '
     f'(«3 дні в стац» тощо) і мовчки стали NaN. Це середнє рахується лише за '
     f'{int(т_числа.notna().sum())} «голими» числами — підмножиною, яка могла відрізнятися '
     f'від решти, тож воно не описує всю вибірку.'),
    ('ПІП', 'мода (top у describe)', опис_піп['top'],
     f'Кожне з {int(опис_піп["unique"])} значень зустрічається {int(опис_піп["freq"])} раз: '
     f'«найчастіше» тут — просто перший у списку. Ідентифікатор не має розподілу.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,середнє,1.496,"Коди 1 і 2 — мітки станів, а не кількості. Результат 1,4..."
1,ступінь КТ (коди 1–5),середнє,2.48,Ступінь — порядкова шкала: відстань від легкого до серед...
2,Дата розтину,коефіцієнт варіації (σ/μ),0.00162,Дата — інтервальна шкала з умовним нулем. Якщо відрахува...
3,"Т вище 37,5","середнє після to_numeric(errors=""coerce"")",3.47,Із 78 непорожніх записів 48 були текстом («3 дні в стац»...
4,ПІП,мода (top у describe),Федорюк Петро Юрійович,Кожне з 121 значень зустрічається 1 раз: «найчастіше» ту...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [39]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × помер/жив

Стать: шкала — не у вашому списку
помер/жив: шкала — номінальна

  Стать  помер/жив
0     ч          1
1     ч          2
2     ж          2
3     ж          2
4     ж          2
5     ж          1


Стать не входить до вісімки, тож її шкалу визначаємо самостійно: номінальна (бінарна) — дві категорії без порядку. помер/жив — номінальна (бінарна). Пара номінальна × номінальна → χ² і V Крамера. Сира Стать має чотири написання (ж, жін, ч, чол) — їх потрібно звести до двох, інакше таблиця спряженості буде 4×2 і хибною.

In [40]:
ОБРАНА_МІРА = 'χ² Пірсона та коефіцієнт Крамера (V)'
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні номінальні (стать і результат — категорії без порядку й відстаней), '
                  'тому єдине допустиме — порівнювати розподіл у таблиці спряженості: χ² перевіряє '
                  'незалежність, V Крамера дає силу зв’язку від 0 до 1.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy import stats

таблиця = pd.crosstab(дані['Стать'], дані['помер/жив'])
частки = таблиця.div(таблиця.sum(axis=1), axis=0)
print('Таблиця спряженості (рядки — стать, стовпці — результат):')
print(таблиця, '\n')
print('Частки за рядками:')
print(частки.round(3), '\n')

хі2, p, ступені, очікувані = stats.chi2_contingency(таблиця, correction=False)
n = таблиця.values.sum()
V = np.sqrt(хі2 / (n * (min(таблиця.shape) - 1)))

print(f'n = {n}; найменша очікувана частота = {очікувані.min():.1f} (умова χ²: ≥ 5 — виконана)')
print(f'χ² = {хі2:.3f}, ступенів свободи = {ступені}, p = {p:.3f}')
print(f'V Крамера = {V:.3f}')
print()
print('Інтерпретація: V ≈ 0,18 — слабкий зв’язок; p ≈ 0,05 — на межі значущості. Вибірка зібрана '
      'майже порівну за результатом (61 : 60), тож частки померлих (57 % жінок, 39 % чоловіків) '
      'НЕ є летальністю в популяції, а причинності ці числа не доводять.')

Таблиця спряженості (рядки — стать, стовпці — результат):
помер/жив  жив  помер
Стать                
ж           32     43
ч           28     18 

Частки за рядками:
помер/жив    жив  помер
Стать                  
ж          0.427  0.573
ч          0.609  0.391 

n = 121; найменша очікувана частота = 22.8 (умова χ²: ≥ 5 — виконана)
χ² = 3.779, ступенів свободи = 1, p = 0.052
V Крамера = 0.177

Інтерпретація: V ≈ 0,18 — слабкий зв’язок; p ≈ 0,05 — на межі значущості. Вибірка зібрана майже порівну за результатом (61 : 60), тож частки померлих (57 % жінок, 39 % чоловіків) НЕ є летальністю в популяції, а причинності ці числа не доводять.


In [41]:
y = df['помер/жив']                   
сирі_коди = {
    'сирі мітки, за порядком появи (factorize)':  pd.factorize(df['Стать'])[0],
    'сирі мітки за алфавітом: ж,жін,ч,чол = 0..3': df['Стать'].map({'ж': 0, 'жін': 1, 'ч': 2, 'чол': 3}),
    'сирі мітки інакше: ч,чол,ж,жін = 0..3':       df['Стать'].map({'ч': 0, 'чол': 1, 'ж': 2, 'жін': 3}),
    'очищена стать, ж=0, ч=1':                      дані['Стать'].map({'ж': 0, 'ч': 1}),
    'очищена стать, ч=0, ж=1':                      дані['Стать'].map({'ч': 0, 'ж': 1}),
}
for назва, к in сирі_коди.items():
    r = np.corrcoef(np.asarray(к, dtype=float), y)[0, 1]
    print(f'{назва:<48} r = {r:+.3f}')
print()
print(f'Для порівняння: V Крамера = {V:.3f}')
print()
print('Висновок: Пірсон вимагає, щоб різниця між кодами означала реальну відстань. Тут коди — '
      'довільні номери, тож для чотирьох сирих міток отримуємо три різні числа (−0,074; +0,101; '
      '+0,248) і навіть різні знаки для тих самих людей — число визначається нашим вибором кодів, '
      'а не даними. Після зведення до двох категорій |r| збігається з φ = V (0,177), '
      'але знак усе одно залежить від того, яку стать назвали «1»: напрямку тут немає, '
      'бо немає порядку.')

сирі мітки, за порядком появи (factorize)        r = +0.101
сирі мітки за алфавітом: ж,жін,ч,чол = 0..3      r = +0.248
сирі мітки інакше: ч,чол,ж,жін = 0..3            r = -0.074
очищена стать, ж=0, ч=1                          r = +0.177
очищена стать, ч=0, ж=1                          r = -0.177

Для порівняння: V Крамера = 0.177

Висновок: Пірсон вимагає, щоб різниця між кодами означала реальну відстань. Тут коди — довільні номери, тож для чотирьох сирих міток отримуємо три різні числа (−0,074; +0,101; +0,248) і навіть різні знаки для тих самих людей — число визначається нашим вибором кодів, а не даними. Після зведення до двох категорій |r| збігається з φ = V (0,177), але знак усе одно залежить від того, яку стать назвали «1»: напрямку тут немає, бо немає порядку.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [42]:
ПІДГОТОВКА = {
    'ПІП': 'Не брати в модель: ідентифікатор не несе величини (і є персональними даними — прибрати або псевдонімізувати).',
    'помер/жив': 'Номінальна змінна: явний словник у 0/1 (помер = 1); one-hot для двох класів зайвий — стовпець був би дзеркальним.',
    'ступінь КТ': 'Звести 12 написань до 5 рівнів і закодувати цілими 1–5 із заданим порядком; 79 пропусків позначити окремим індикатором «КТ не описано».',
    'Дата розтину': 'Дату саму не брати; перейти до тривалості «розтин − поступлення» у днях. Для прогнозу виключити: заповнена лише в померлих — витік мети.',
    'КТ': 'Розібрати текст у частку 0–1 (діапазон — середина), додати ознаку способу запису; потім min–max, бо шкала обмежена 0..1.',
    'Т вище 37,5': 'Вийняти число днів із тексту; 43 пропуски не заповнювати нулями (невідомо, чи «не було», чи «не записано») — додати індикатор.',
    'Сечов1': 'Логарифмувати (правий хвіст: 60 і 68 при медіані 7,2), потім стандартизувати (z-оцінка); пропуски не заповнювати мовчки.',
    'БРпр1': 'Мін–макс нормалізація (стовпець змінний, max ≠ min); значення дискретні з кроком 1,7 — це не заважає, але не вважати їх безперервними.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,Не брати в модель: ідентифікатор не несе величини (і є п...
1,помер/жив,номінальна,Номінальна змінна: явний словник у 0/1 (помер = 1); one-...
2,ступінь КТ,порядкова,Звести 12 написань до 5 рівнів і закодувати цілими 1–5 і...
3,Дата розтину,інтервальна,Дату саму не брати; перейти до тривалості «розтин − пост...
4,КТ,відношень,"Розібрати текст у частку 0–1 (діапазон — середина), дода..."
5,"Т вище 37,5",відношень,Вийняти число днів із тексту; 43 пропуски не заповнювати...
6,Сечов1,відношень,"Логарифмувати (правий хвіст: 60 і 68 при медіані 7,2), п..."
7,БРпр1,відношень,"Мін–макс нормалізація (стовпець змінний, max ≠ min); зна..."


In [43]:
готово = pd.DataFrame(index=df.index)

готово['померла'] = (дані['помер/жив'] == 'помер').astype(int)

готово['ступінь КТ, 1–5'] = дані['ступінь КТ'].cat.codes.where(lambda s: s >= 0) + 1
готово['КТ описано'] = дані['ступінь КТ'].notna().astype(int)

готово['днів до розтину'] = (df['Дата розтину'] - df['Поступлення']).dt.days

лог = np.log1p(df['Сечов1'])
готово['Сечов1, log-z'] = (лог - лог.mean()) / лог.std()

for звідки, куди in [(df['БРпр1'], 'БРпр1, 0–1'), (дані['КТ частка'], 'КТ частка, 0–1')]:
    assert звідки.max() != звідки.min()
    готово[куди] = (звідки - звідки.min()) / (звідки.max() - звідки.min())

готово['Т дні'] = дані['Т дні']

print(готово.describe().T[['count', 'mean', 'std', 'min', 'max']].round(3))
print()
print('Дати розтину раніше за поступлення (імовірні помилки введення дат):',
      int((готово['днів до розтину'] < 0).sum()), '— їх треба перевірити, а не нормалізувати')

                 count    mean     std      min     max
померла          121.0   0.504   0.502    0.000   1.000
ступінь КТ, 1–5   42.0   2.476   1.110    1.000   5.000
КТ описано       121.0   0.347   0.478    0.000   1.000
днів до розтину   61.0  10.197  48.961 -343.000  80.000
Сечов1, log-z    117.0   0.000   1.000   -1.739   4.774
БРпр1, 0–1       109.0   0.308   0.202    0.000   1.000
КТ частка, 0–1    43.0   0.500   0.293    0.000   1.000
Т дні             78.0   4.218   2.399    1.000  14.000

Дати розтину раніше за поступлення (імовірні помилки введення дат): 2 — їх треба перевірити, а не нормалізувати


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [44]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


Ця перевірка нічого не знайшла — і це не означає, що все гаразд: `to_numeric(errors='coerce')`
мовчки перетворює текст на NaN і ховає справжню проблему. Спробуємо формулу
`(x − min) / (max − min)` до сирих стовпців.

In [ ]:
def мінмакс_наївно(s):
    return (s - s.min()) / (s.max() - s.min())

результат_нормалізації = {}
for c in МОЇ:
    try:
        r = мінмакс_наївно(df[c].dropna())
        результат_нормалізації[c] = r
        print(f'{c:<14} працює, діапазон {r.min():.2f}–{r.max():.2f}')
    except Exception as e:
        print(f'{c:<14} ПАДАЄ: {type(e).__name__}: {str(e)[:70]}')

сеч = результат_нормалізації['Сечов1']
ЧАСТКА_СЕЧОВ_ДО_01 = (сеч < 0.1).mean()
print()
print(f'Сечов1: {ЧАСТКА_СЕЧОВ_ДО_01:.0%} значень потрапляють у проміжок 0–0,1 через два викиди (60 і 68).')

ПІП            ПАДАЄ: TypeError: unsupported operand type(s) for -: 'str' and 'str'
помер/жив      працює, діапазон 0.00–1.00
ступінь КТ     ПАДАЄ: TypeError: unsupported operand type(s) for -: 'str' and 'str'
Дата розтину   працює, діапазон 0.00–1.00
КТ             ПАДАЄ: TypeError: '<=' not supported between instances of 'str' and 'float'
Т вище 37,5    ПАДАЄ: TypeError: '<=' not supported between instances of 'int' and 'str'
Сечов1         працює, діапазон 0.00–1.00
БРпр1          працює, діапазон 0.00–1.00

Сечов1: 75% значень потрапляють у проміжок 0–0,1 через два викиди (60 і 68).


In [46]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.


In [47]:
ВИПРАВЛЕННЯ = [
    ('1',
     'Середнє арифметичне для номінальної змінної «помер/жив» (коди 1 і 2).',
     'Коди — мітки двох станів, а не кількості. 1,50 лише відображає, що померлих і живих майже порівну '
     '(61 і 60); «рівно посередині між летальним і сприятливим» — вигадка: такого стану не існує.',
     'Лише частоти й частки: померло 61 із 121 (50,4 %), жили 60 (49,6 %). Якщо потрібна одна цифра — '
     'частка померлих, тобто середнє індикатора 0/1.'),
    ('2',
     'Дані не придатні: одна змінна записана двома парами несумісних кодувань (ж/жін, ч/чол).',
     'value_counts() рахує написання, а не людей: «чотири статі» — артефакт запису. Насправді жінок '
     '75 (56 + 19), чоловіків 46 (34 + 12); «групи жін і чол помітно менші» теж хибне.',
     'Спершу звести до двох значень (replace({"жін": "ж", "чол": "ч"})), перевірити, що лишилося два, '
     'і лише потім рахувати частоти: ж — 75 (62 %), ч — 46 (38 %).'),
    ('3',
     'Порядкова шкала «ступінь КТ»: сортування за алфавітом, середнє номерів і брудні дані.',
     'sort_index() сортує рядки за абеткою, тому «найлегший — важкий», «найважчий — середній». Є дублі через '
     'пробіл («важкий» і «важкий »), помилки («сердній», «серед ступ»). Середнє номерів 5,9 → «легкий» '
     'безглузде: відстані між ступенями невідомі й нерівні.',
     'Виправити написання (12 → 5 значень), задати порядок «легкий < середній < важкий < вкрай важкий < '
     'критичний» (ordered Categorical). Допустимі: частоти, мода, медіана — «середній»; середнього немає.'),
    ('4',
     'Дані не придатні: pd.to_numeric(errors="coerce") мовчки відкидає текстові записи КТ.',
     'З 43 придатних записів 16 були діапазонами чи межею («20-30%», «до 17%») і стали NaN; рахується лише '
     '27 «чистих» часток. Відкинуті значення систематично більші (діапазони ≈ 49,5 % проти 39,6 %), тож '
     '39,6 % — занижена оцінка зі зсувом.',
     'Розібрати текст у частки (діапазон — середина, «до 17%» — межа) і записати, скільки значень наближені. '
     'Тоді n = 43, середнє ≈ 42,5 %, медіана 35 %. Результат подавати з позначкою про наближення.'),
    ('5',
     'Дані не придатні: одна й та сама величина в двох кодуваннях — значення 0,9 серед відсотків.',
     'Сатурація 0,9 % несумісна з життям; це, найімовірніше, 0,9 як частка (90 %). Мінімум «0,9» неправдивий, а '
     'середнє 82,05 занижене: після виправлення 82,8; медіана 84 від викиду не залежить.',
     'Звести до одиниць (частка × 100 або виключити запис із поясненням). Висновок про «більшість» '
     'підтверджувати часткою, а не середнім: SpO2 нижче 90 % у 74 % (90 із 120) пацієнтів.'),
    ('6',
     'Коефіцієнт варіації для дат (інтервальна шкала з умовним нулем).',
     'CV = σ/μ залежить від того, звідки відлічувати дні: від 1970 року виходить 0,595 %, від нуля Excel — '
     'інше число. «Однорідність потоку» — артефакт точки відліку; на шкалі без природного нуля відношення '
     'не визначені.',
     'Для дат — різниці: розмах 463 дні (25.02.2021 – 03.06.2022), σ ≈ 113 днів, медіанна дата. Однорідність '
     'потоку оцінювати кількістю поступлень за тиждень чи місяць. CV для дат відкинути.'),
    ('7',
     'Кореляція Пірсона між двома номінальними змінними; а також висновок «r близьке до нуля».',
     'Стать і результат — категорії без порядку; Пірсон вимагає відстаней між кодами. Число 0,177 залежить '
     'від довільного вибору, кого назвати 1, а не 0. Воно й не «близьке до нуля»: це слабкий, але ненульовий '
     'зв’язок (p ≈ 0,05), тож «не виявлено» — безпідставне.',
     'Таблиця спряженості, χ² і V Крамера: χ² = 3,78, p = 0,052, V = 0,177. Правильний висновок: слабкий зв’язок на межі значущості; відсутність не доведена.'),
    ('8',
     'Нормалізація без перевірки шкал і варіативності: «усі числові» ознаки поспіль.',
     'ФЕРИТ2 має єдине значення: max = min, ділення 0/0 дає порожній стовпець (count = 0). ФЕРИТ1 має два '
     'спостереження (0 і 1). «помер/жив» — номінальні коди, їх нормалізація — дія над мітками. Тож '
     '«усі успішно нормалізовано» хибне; а КТ, «Т вище 37,5», «АТ пост» узагалі пропущено як текст.',
     'Спершу очистити текст у числа, потім вибрати лише змінні відношень/інтервальні з достатнім n та '
     'розкидом (nunique > 2, max ≠ min). Сталі стовпці виключити, номінальні кодувати one-hot, '
     'ідентифікатори не брати.'),
    ('9',
     'Дані не придатні для dropna(): пропуски структурні, а висновок суперечить виводу.',
     'Результат — 0 рядків із 121, а у звіті написано «повністю заповнений набір». Пропуски не випадкові: '
     '«Дата розтину» порожня рівно у 60 живих (не застосовується), ФЕРИТ2 порожній на 99 %, ІЛ6-2 — на 95 %.',
     'Не видаляти рядки наосліп: оцінити частку пропусків за стовпцями, виключити стовпці майже без даних, '
     'структурні пропуски позначити індикатором. Тоді 35 стовпців із ≤ 10 % пропусків дають 91 повний рядок.'),
    ('10',
     'Витік мети й кореляція Пірсона (рейтинг за |r|) на непридатних даних.',
     '«Дата розтину» існує лише в померлих; astype("int64") перетворює порожні дати живих на −9,2·10¹⁸ — '
     'тому |r| = 1,00. Це не предиктор, а наслідок мети. Також ІЛ6-2 (|r| = 0,62) пораховано на 6 '
     'спостереженнях, а знак у рейтингу втрачено через abs().',
     'Виключити «Дату розтину» (витік) та ідентифікатори; рахувати лише для стовпців із n ≥ 60 і показувати n '
     'і знак. Тоді лідирують Лім2 (r = −0,65, n = 104), Л2 (0,48), Вік (0,45); Лім2 і Л2 — повторні аналізи, '
     'тож й вони можуть бути наслідком тяжкого перебігу.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє арифметичне для номінальної змінної «помер/жив» ...,"Коди — мітки двох станів, а не кількості. 1,50 лише відо...","Лише частоти й частки: померло 61 із 121 (50,4 %), жили ..."
1,2,Дані не придатні: одна змінна записана двома парами несу...,"value_counts() рахує написання, а не людей: «чотири стат...","Спершу звести до двох значень (replace({""жін"": ""ж"", ""чол..."
2,3,"Порядкова шкала «ступінь КТ»: сортування за алфавітом, с...","sort_index() сортує рядки за абеткою, тому «найлегший — ...","Виправити написання (12 → 5 значень), задати порядок «ле..."
3,4,"Дані не придатні: pd.to_numeric(errors=""coerce"") мовчки ...",З 43 придатних записів 16 були діапазонами чи межею («20...,"Розібрати текст у частки (діапазон — середина, «до 17%» ..."
4,5,Дані не придатні: одна й та сама величина в двох кодуван...,"Сатурація 0,9 % несумісна з життям; це, найімовірніше, 0...",Звести до одиниць (частка × 100 або виключити запис із п...
5,6,Коефіцієнт варіації для дат (інтервальна шкала з умовним...,"CV = σ/μ залежить від того, звідки відлічувати дні: від ...",Для дат — різниці: розмах 463 дні (25.02.2021 – 03.06.20...
6,7,Кореляція Пірсона між двома номінальними змінними; а так...,Стать і результат — категорії без порядку; Пірсон вимага...,"Таблиця спряженості, χ² і V Крамера: χ² = 3,78, p = 0,05..."
7,8,Нормалізація без перевірки шкал і варіативності: «усі чи...,"ФЕРИТ2 має єдине значення: max = min, ділення 0/0 дає по...","Спершу очистити текст у числа, потім вибрати лише змінні..."
8,9,"Дані не придатні для dropna(): пропуски структурні, а ви...","Результат — 0 рядків із 121, а у звіті написано «повніст...",Не видаляти рядки наосліп: оцінити частку пропусків за с...
9,10,Витік мети й кореляція Пірсона (рейтинг за |r|) на непри...,"«Дата розтину» існує лише в померлих; astype(""int64"") пе...",Виключити «Дату розтину» (витік) та ідентифікатори; раху...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [48]:

from scipy import stats

print('Ступінь КТ')
ст_хиб = df['ступінь КТ'].value_counts().sort_index()
середнє_хиб = df['ступінь КТ'].map({n: i for i, n in enumerate(ст_хиб.index)}).mean()
print(f'ХИБНО     : {len(ст_хиб)} «ступенів», «середній» = {середнє_хиб:.1f} → {ст_хиб.index[int(round(середнє_хиб))]!r}; '
      f'«найлегший» = {ст_хиб.index[0]!r}')
рівні = дані['ступінь КТ'].dropna().sort_values().reset_index(drop=True)
медіана_КТ = рівні.iloc[(len(рівні) - 1) // 2]
print('ПРАВИЛЬНО :', дані['ступінь КТ'].value_counts(sort=False).to_dict(), '| медіана:', медіана_КТ)
assert медіана_КТ == 'середній'

print('\nКТ у відсотках')
кт_хиб = pd.to_numeric(df['КТ'], errors='coerce')
кт_пр = дані['КТ частка']
print(f'ХИБНО     : n = {кт_хиб.notna().sum()}, середнє = {100*кт_хиб.mean():.1f} %, медіана = {100*кт_хиб.median():.1f} %')
print(f'ПРАВИЛЬНО : n = {кт_пр.notna().sum()}, середнє = {100*кт_пр.mean():.1f} %, медіана = {100*кт_пр.median():.1f} %')
print('            середнє за способом запису:', (100 * кт_пр.groupby(дані['КТ спосіб']).mean()).round(1).to_dict())
assert кт_пр.notna().sum() == 43 and кт_хиб.notna().sum() == 27

print('\nДати поступлення')
дати_п = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
for нуль in ('1970-01-01', '1899-12-30'):
    дні = (дати_п - pd.Timestamp(нуль)).dt.days
    print(f'ХИБНО     : CV від {нуль} = {100 * дні.std() / дні.mean():.3f} %')
print(f'ПРАВИЛЬНО : розмах {(дати_п.max() - дати_п.min()).days} днів, σ = {дати_п.std().days} днів, '
      f'медіанна дата {дати_п.median().date()}')

print('\nСтать × результат')
r_хиб = df['Стать'].astype(str).str.strip().str[0].map({'ч': 0, 'ж': 1}).corr((df['помер/жив'] == 1).astype(int))
print(f'ХИБНО     : Пірсон r = {r_хиб:.3f} → «зв’язку немає»')
print(f'ПРАВИЛЬНО : χ² = {хі2:.2f}, p = {p:.3f}, V = {V:.3f} → слабкий зв’язок на межі значущості')

print('\nПропуски')
частка_проп = df.isna().mean().sort_values(ascending=False)
print(f'ХИБНО     : dropna() → {len(df.dropna())} рядків із {len(df)}')
print('ПРАВИЛЬНО : найбільше пропусків —', частка_проп.head(4).round(2).to_dict())
print('            «Дата розтину» порожня серед померлих:', int(df.loc[df['помер/жив'] == 1, 'Дата розтину'].isna().sum()),
      '| серед живих:', int(df.loc[df['помер/жив'] == 2, 'Дата розтину'].isna().sum()))
придатні = частка_проп[частка_проп <= 0.10].index
print(f'            стовпців із ≤ 10 % пропусків: {len(придатні)}, повних рядків у них: {len(df[придатні].dropna())}')
assert len(df.dropna()) == 0 and len(df[придатні].dropna()) == 91

print('\nРейтинг ознак')
НТ_МІН = np.iinfo('int64').min
розтин_число = pd.to_datetime(df['Дата розтину'], errors='coerce').map(lambda t: t.value if pd.notna(t) else НТ_МІН)
ціль = (df['помер/жив'] == 1).astype(int)
print(f'ХИБНО     : |r| для «Дати розтину» = {abs(np.corrcoef(розтин_число.astype(float), ціль)[0, 1]):.3f} (порожні дати → {НТ_МІН:.1e})')
виключити = {'номер і/х', 'Телефон', 'Дата розтину', 'помер/жив'}
рядки = []
for c in df.select_dtypes('number'):
    маска = df[c].notna()
    if c in виключити or маска.sum() < 60 or df.loc[маска, c].nunique() < 3:
        continue
    r_, p_ = stats.pointbiserialr(ціль[маска], df.loc[маска, c])
    рядки.append((c, round(r_, 3), int(маска.sum()), round(p_, 4)))
рейтинг_пр = pd.DataFrame(рядки, columns=['ознака', 'r', 'n', 'p']).sort_values('r', key=abs, ascending=False)
print('ПРАВИЛЬНО : без витоку й ознак із малим n (з r зі знаком і n):')
print(рейтинг_пр.head(5).to_string(index=False))

Ступінь КТ
ХИБНО     : 12 «ступенів», «середній» = 5.9 → 'легкий'; «найлегший» = 'важкий'
ПРАВИЛЬНО : {'легкий': 7, 'середній': 18, 'важкий': 10, 'вкрай важкий': 4, 'критичний': 3} | медіана: середній

КТ у відсотках
ХИБНО     : n = 27, середнє = 39.6 %, медіана = 30.0 %
ПРАВИЛЬНО : n = 43, середнє = 42.5 %, медіана = 35.0 %
            середнє за способом запису: {'верхня межа': 17.0, 'діапазон': 49.5, 'немає': nan, 'точно': 39.6}

Дати поступлення
ХИБНО     : CV від 1970-01-01 = 0.595 %
ХИБНО     : CV від 1899-12-30 = 0.254 %
ПРАВИЛЬНО : розмах 463 днів, σ = 113 днів, медіанна дата 2022-02-27

Стать × результат
ХИБНО     : Пірсон r = 0.177 → «зв’язку немає»
ПРАВИЛЬНО : χ² = 3.78, p = 0.052, V = 0.177 → слабкий зв’язок на межі значущості

Пропуски
ХИБНО     : dropna() → 0 рядків із 121
ПРАВИЛЬНО : найбільше пропусків — {'ФЕРИТ2': 0.99, 'ФЕРИТ1': 0.98, 'ІЛ6-2': 0.95, 'КТ опис': 0.92}
            «Дата розтину» порожня серед померлих: 0 | серед живих: 60
            стовпців із ≤ 10 % п

---
# Етап 7. Висновки та індивідуальне питання

In [49]:
ВИСНОВКИ = [
    (f'Три з восьми стовпців потребують очищення ще до будь-яких обчислень: «ступінь КТ» має '
     f'{N_СТУПІНЬ_СИРИХ} написань замість 5 рівнів, «КТ» — {N_КТ_ТЕКСТ} текстових записів із {N_КТ_НЕПОР}, '
     f'«Т вище 37,5» — {N_Т_ТЕКСТ} текстових із {N_Т_НЕПОР}, водночас «помер/жив» має тип int64, '
     f'хоч насправді номінальна, тож тип даних шкалу не визначає.'),
    (f'«Дата розтину» заповнена в усіх 61 померлих і в жодного з 60 живих, тож це не ознака, а наслідок '
     f'цільової змінної (витік): у прогнозній моделі її й похідну «днів до розтину» використовувати не можна; '
     f'до того ж «ступінь КТ» пропущено в {N_СТУПІНЬ_ПРОПУСКІВ} з 121 пацієнтів.'),
    (f'Зв’язок «Стать × помер/жив» (обидві номінальні) слабкий: χ² = {хі2:.2f}, p = {p:.3f}, V = {V:.2f}; '
     f'p лежить на межі значущості , '
     f'а вибірка майже збалансована за результатом (61 : 60), тож частки не є летальністю.'),
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Три з восьми стовпців потребують очищення ще до будь-яких обчислень: «ступінь КТ» має 12 написань замість 5 рівнів, «КТ» — 17 текстових записів із 44, «Т вище 37,5» — 48 текстових із 78, водночас «помер/жив» має тип int64, хоч насправді номінальна, тож тип даних шкалу не визначає.
2. «Дата розтину» заповнена в усіх 61 померлих і в жодного з 60 живих, тож це не ознака, а наслідок цільової змінної (витік): у прогнозній моделі її й похідну «днів до розтину» використовувати не можна; до того ж «ступінь КТ» пропущено в 79 з 121 пацієнтів.
3. Зв’язок «Стать × помер/жив» (обидві номінальні) слабкий: χ² = 3.78, p = 0.052, V = 0.18; p лежить на межі значущості , а вибірка майже збалансована за результатом (61 : 60), тож частки не є летальністю.


In [50]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    f'Формула (x − min)/(max − min) ламається на стовпці «Т вище 37,5» (так само на «КТ»). Це стовпець '
    f'типу object: {N_Т_ТЕКСТ} із {N_Т_НЕПОР} записів — текст на кшталт «3 дні в стац» і «5 до гос», '
    f'решта — цілі числа. Вираз s.min() або s − s.min() кидає TypeError, бо не можна порівнювати чи віднімати '
    f'рядок і число. Якщо ж «полагодити» це через pd.to_numeric(errors="coerce"), помилки не буде, але {N_Т_ТЕКСТ} '
    f'із {N_Т_НЕПОР} значень мовчки стануть NaN, і нормалізація охопить лише «голі» числа.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.

Формула (x − min)/(max − min) ламається на стовпці «Т вище 37,5» (так само на «КТ»). Це стовпець типу object: 48 із 78 записів — текст на кшталт «3 дні в стац» і «5 до гос», решта — цілі числа. Вираз s.min() або s − s.min() кидає TypeError, бо не можна порівнювати чи віднімати рядок і число. Якщо ж «полагодити» це через pd.to_numeric(errors="coerce"), помилки не буде, але 48 із 78 значень мовчки стануть NaN, і нормалізація охопить лише «голі» числа.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [51]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'код, перевірено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'код, перевірено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірила всі коди, які згенерував штучний інтелект, тексти по більшій мірі писала сама, проте могла взяти в ші ідею для напрямку'

ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [52]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 7 готова до захисту.
Олійник Божена, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.